In [1]:
import os

In [2]:
%pwd

'c:\\Users\\KiranSreehari\\Desktop\\text summarizer\\Text-Summarizer-\\research'

In [3]:
os.chdir("../")

In [4]:
%pwd

'c:\\Users\\KiranSreehari\\Desktop\\text summarizer\\Text-Summarizer-'

In [5]:
from dataclasses import dataclass
from pathlib import Path

@dataclass(frozen=True)
class ModelTrainerConfig:
    root_dir: Path
    data_path: Path
    model_ckpt: Path
    num_train_epochs: int
    warmup_steps: int
    per_device_train_batch_size: int
    weight_decay: float
    logging_steps: int
    evaluation_strategy: str
    eval_steps: int
    save_steps: float
    gradient_accumulation_steps: int

In [6]:
from textsummarizer.constants import *
from textsummarizer.utils.common import read_yaml, create_directories
import torch

In [7]:
import os
print(os.getcwd())

c:\Users\KiranSreehari\Desktop\text summarizer\Text-Summarizer-


In [8]:
import sys
sys.path.append("..")

In [9]:
import os

print(os.listdir(".."))

['Text-Summarizer-']


In [10]:
import sys
import os

project_root = os.path.abspath("..")

if project_root not in sys.path:
    sys.path.insert(0, project_root)

print(project_root)

c:\Users\KiranSreehari\Desktop\text summarizer


In [11]:
from src.textsummarizer.config.configuration import ModelTrainerConfig

In [19]:
class ModelTrainer:
    def __init__(self, config: ModelTrainerConfig):
        self.config = config



    def train(self):
        device = "cuda" if torch.cuda.is_available() else "cpu"
        tokenizer = AutoTokenizer.from_pretrained("./tokenizer")
        model_pegasus = AutoModelForSeq2SeqLM.from_pretrained(self.config.model_ckpt).to(device)
        seq2seq_data_collator = DataCollatorForSeq2Seq(tokenizer, model = model_pegasus)


        ##loading data
        dataset_samsum_pt = load_from_disk(self.config.data_path)



        #trainer_args = TrainingArguments(
        #    output_dir = self.config.root_dir, num_train_epochs = self.config.num_train_epochs, warmup_steps = self.config.warmup_steps,
         #   per_device_train_batch_size = self.config.per_device_train_batch_size, per_device_eval_batch_size = self.config.per_device_train_batch_size,
          #  weight_decay = self.config.weight_decay, logging_steps = self.config.logging_steps, evaluation_strategy = self.config.evaluation_strategy,
           # eval_steps = self.config.eval_steps, save_steps =1e6, gradient_accumulation_steps = self.config.gradient_accumulation_steps)



        trainer_args = TrainingArguments(
            output_dir = self.config.root_dir, num_train_epochs = 1, warmup_steps = 500,
            per_device_train_batch_size = 1, per_device_eval_batch_size = 1, weight_decay = 0.01, 
            logging_steps = 10, evaluation_strategy = 'steps', eval_steps = 500, save_steps = 1e6,
            gradient_accumulation_steps = 16)




        trainer = Trainer(
            model=model_pegasus,
            args=trainer_args,
            tokenizer=tokenizer,
            data_collator=seq2seq_data_collator,
            train_dataset=dataset_samsum_pt["test"],
            eval_dataset=dataset_samsum_pt["validation"]
)

        trainer.train()

        ##save model
        model_pegasus.save_pretrained(os.path.join(self.config.root_dir, "pegasus-samsum-model"))

        ##save tokenizer
        tokenizer.save_pretrained(os.path.join(self.config.root_dir, "tokenizer"))






In [20]:
try:
    config = ConfigurationManager()

    model_trainer_config = config.get_model_trainer_config()

    model_trainer = ModelTrainer(config=model_trainer_config)

    model_trainer.train()

except Exception as e:
    raise e

FileNotFoundError: [Errno 2] No such file or directory: '..\\config\\config.yaml'

In [17]:
import sys
import os

sys.path.insert(0, os.path.abspath(".."))

from src.textsummarizer.config.configuration import ConfigurationManager

In [18]:
from src.textsummarizer.components.model_trainer import ModelTrainer

ModuleNotFoundError: No module named 'src.textsummarizer.components.model_trainer'

In [21]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

tokenizer = AutoTokenizer.from_pretrained("./tokenizer")
model = AutoModelForSeq2SeqLM.from_pretrained("./pegasus-samsum-model")

print("Model loaded successfully")
print("Tokenizer loaded successfully")

c:\Users\KiranSreehari\Desktop\text summarizer\Text-Summarizer-\textS\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Model loaded successfully
Tokenizer loaded successfully


In [22]:
text = """
The company announced that it will launch a new electric vehicle next year.
The vehicle will have improved battery performance and advanced safety features.
The company expects the new model to attract more customers and increase sales.
"""

inputs = tokenizer(
    text,
    return_tensors="pt",
    truncation=True,
    max_length=512
)

summary_ids = model.generate(
    inputs["input_ids"],
    max_length=64,
    num_beams=4,
    early_stopping=True
)

summary = tokenizer.decode(
    summary_ids[0],
    skip_special_tokens=True
)

print("Summary:")
print(summary)

Summary:
The company expects the new model to attract more customers and increase sales .<n>The vehicle will have improved battery performance and advanced safety features .<n>The company expects the new model to attract more customers and increase sales .


In [23]:
try:
    config = ConfigurationManager()

    model_trainer_config = config.get_model_trainer_config()

    model_trainer = ModelTrainer(config=model_trainer_config)

    model_trainer.train()

except Exception as e:
    raise e

FileNotFoundError: [Errno 2] No such file or directory: '..\\config\\config.yaml'

In [24]:
from pathlib import Path

from src.textsummarizer.constants import CONFIG_FILE_PATH, PARAMS_FILE_PATH

print("Config path:", CONFIG_FILE_PATH)
print("Params path:", PARAMS_FILE_PATH)

Config path: ..\config\config.yaml
Params path: ..\params.yaml


In [25]:
import importlib
import textsummarizer.constants

importlib.reload(textsummarizer.constants)

from textsummarizer.constants import CONFIG_FILE_PATH, PARAMS_FILE_PATH

print("Config path:", CONFIG_FILE_PATH)
print("Params path:", PARAMS_FILE_PATH)

Config path: config\config.yaml
Params path: params.yaml


In [26]:
try:
    config = ConfigurationManager()

    model_trainer_config = config.get_model_trainer_config()

    model_trainer = ModelTrainer(config=model_trainer_config)

    model_trainer.train()

except Exception as e:
    raise e

FileNotFoundError: [Errno 2] No such file or directory: '..\\config\\config.yaml'

In [27]:
import importlib

import textsummarizer.constants
import textsummarizer.config.configuration

importlib.reload(textsummarizer.constants)
importlib.reload(textsummarizer.config.configuration)

from textsummarizer.config.configuration import ConfigurationManager

print("ConfigurationManager reloaded")

ConfigurationManager reloaded


In [28]:
try:
    config = ConfigurationManager()

    model_trainer_config = config.get_model_trainer_config()

    model_trainer = ModelTrainer(config=model_trainer_config)

    model_trainer.train()

except Exception as e:
    raise e

[2026-10-05 23:53:22,130: INFO: common: yaml file: config\config.yaml loaded successfully]
[2026-10-05 23:53:22,134: INFO: common: yaml file: params.yaml loaded successfully]
[2026-10-05 23:53:22,136: INFO: common: created directory at: artifacts]
[2026-10-05 23:53:22,138: INFO: common: created directory at: artifacts/model_trainer]


NameError: name 'DataCollatorForSeq2Seq' is not defined

In [29]:
from transformers import DataCollatorForSeq2Seq

In [30]:
try:
    config = ConfigurationManager()

    model_trainer_config = config.get_model_trainer_config()

    model_trainer = ModelTrainer(config=model_trainer_config)

    model_trainer.train()

except Exception as e:
    raise e

[2026-10-05 23:54:22,847: INFO: common: yaml file: config\config.yaml loaded successfully]
[2026-10-05 23:54:22,849: INFO: common: yaml file: params.yaml loaded successfully]
[2026-10-05 23:54:22,850: INFO: common: created directory at: artifacts]
[2026-10-05 23:54:22,851: INFO: common: created directory at: artifacts/model_trainer]


NameError: name 'load_from_disk' is not defined

In [31]:
from datasets import load_from_disk

In [32]:
try:
    config = ConfigurationManager()

    model_trainer_config = config.get_model_trainer_config()

    model_trainer = ModelTrainer(config=model_trainer_config)

    model_trainer.train()

except Exception as e:
    raise e

[2026-10-05 23:54:50,539: INFO: common: yaml file: config\config.yaml loaded successfully]
[2026-10-05 23:54:50,541: INFO: common: yaml file: params.yaml loaded successfully]
[2026-10-05 23:54:50,542: INFO: common: created directory at: artifacts]
[2026-10-05 23:54:50,543: INFO: common: created directory at: artifacts/model_trainer]


NameError: name 'TrainingArguments' is not defined

In [33]:
from transformers import TrainingArguments

In [ ]:
try:
    config = ConfigurationManager()

    model_trainer_config = config.get_model_trainer_config()

    model_trainer = ModelTrainer(config=model_trainer_config)

    model_trainer.train()

except Exception as e:
    raise e

[2026-10-05 23:55:19,595: INFO: common: yaml file: config\config.yaml loaded successfully]
[2026-10-05 23:55:19,597: INFO: common: yaml file: params.yaml loaded successfully]
[2026-10-05 23:55:19,597: INFO: common: created directory at: artifacts]
[2026-10-05 23:55:19,598: INFO: common: created directory at: artifacts/model_trainer]


c:\Users\KiranSreehari\Desktop\text summarizer\Text-Summarizer-\textS\Lib\site-packages\transformers\training_args.py:1568: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(


NameError: name 'Trainer' is not defined

In [35]:
from transformers import Trainer

In [36]:
try:
    config = ConfigurationManager()

    model_trainer_config = config.get_model_trainer_config()

    model_trainer = ModelTrainer(config=model_trainer_config)

    model_trainer.train()

except Exception as e:
    raise e

[2026-10-05 23:55:53,401: INFO: common: yaml file: config\config.yaml loaded successfully]
[2026-10-05 23:55:53,403: INFO: common: yaml file: params.yaml loaded successfully]
[2026-10-05 23:55:53,405: INFO: common: created directory at: artifacts]
[2026-10-05 23:55:53,405: INFO: common: created directory at: artifacts/model_trainer]


C:\Users\KiranSreehari\AppData\Local\Temp\ipykernel_10600\839322395.py:36: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(
 20%|█▉        | 10/51 [09:50<40:52, 59.82s/it]

{'loss': 2.8039, 'grad_norm': 266.8759765625, 'learning_rate': 1.0000000000000002e-06, 'epoch': 0.2}


 39%|███▉      | 20/51 [19:14<28:47, 55.73s/it]

{'loss': 2.7387, 'grad_norm': 173.41307067871094, 'learning_rate': 2.0000000000000003e-06, 'epoch': 0.39}


...

{'loss': 2.7216, 'grad_norm': 352.00921630859375, 'learning_rate': 4.000000000000001e-06, 'epoch': 0.78}


 98%|█████████▊| 50/51 [46:34<00:54, 54.69s/it]

{'loss': 2.6192, 'grad_norm': 692.152587890625, 'learning_rate': 5e-06, 'epoch': 0.98}


100%|██████████| 51/51 [47:46<00:00, 56.20s/it]


{'train_runtime': 2866.1159, 'train_samples_per_second': 0.286, 'train_steps_per_second': 0.018, 'train_loss': 2.7427861877516206, 'epoch': 1.0}
